# RAG

Build a Retrieval-Augmented Generation pipeline for the IDS Knowledge Base using **BGE embeddings** and **FAISS** for semantic search and relevant information retrieval.


## 1. Import Libraries

In [1]:
from pathlib import Path

from langchain_community.document_loaders import TextLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_huggingface import HuggingFaceEmbeddings

from langchain_community.vectorstores import FAISS

import joblib

C:\Users\COMPUMARTS\AppData\Local\Temp\ipykernel_14532\1203002905.py:3: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import TextLoader


In [4]:
import joblib

model = joblib.load("models/rf_model.pkl")
label_encoder = joblib.load("models/label_encoder.pkl")
feature_names = joblib.load("models/feature_names.pkl")
scaler = joblib.load("models/scaler.pkl")

# 2. Load Markdown Files
Load the IDS knowledge base and attach the corresponding attack name as metadata.

In [7]:
knowledge_path = Path("../knowledge_base")

In [8]:
files = list(knowledge_path.glob("*.md"))

print(f"Found {len(files)} files")

for file in files:
    print(file.name)

Found 8 files
BENIGN.md
Bot.md
Brute_Force.md
DDoS.md
DoS.md
PortScan.md
Rare_Attack.md
Web_Attack.md


In [9]:
documents = []

for file in files:
    loader = TextLoader(str(file), encoding="utf-8")

    docs = loader.load()

    attack_name = file.stem

    for doc in docs:
        doc.metadata["attack"] = attack_name

    documents.extend(docs)

# 3. Split into Chunks
Split the knowledge base into overlapping chunks to improve retrieval accuracy.

In [10]:
splitter = RecursiveCharacterTextSplitter(
    chunk_size=800,
    chunk_overlap=150
)

chunks = splitter.split_documents(documents)

print("Chunks:", len(chunks))

Chunks: 56


# 4. Create Embeddings

In [11]:
embeddings = HuggingFaceEmbeddings(
    model_name="BAAI/bge-small-en-v1.5",
    encode_kwargs={"normalize_embeddings": True}
)

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

# 5. Build FAISS Index

In [12]:
vector_db = FAISS.from_documents(
    documents=chunks,
    embedding=embeddings
)

# 6. Save FAISS Index

In [13]:
vector_db.save_local("faiss_index")

In [14]:
print(documents[0].metadata)

{'source': '..\\knowledge_base\\BENIGN.md', 'attack': 'BENIGN'}


# 7. Test Retrieval

In [15]:
test_queries = [
    "DDoS attack indicators of compromise",
    "PortScan MITRE ATT&CK mapping",
    "Brute Force attack detection methods",
    "Web Attack incident response",
    "Bot detection techniques",
    "DoS mitigation strategies",
    "Rare Attack detection"
]

for query in test_queries:
    print("=" * 100)
    print(f"Query: {query}")

    results = vector_db.similarity_search(query, k=2)

    for i, result in enumerate(results, 1):
        print(f"\nResult {i}")
        print(f"Source : {result.metadata['source']}")
        print(f"Attack : {result.metadata['attack']}")
        print("-" * 80)
        print(result.page_content[:300])

Query: DDoS attack indicators of compromise

Result 1
Source : ..\knowledge_base\DoS.md
Attack : DoS
--------------------------------------------------------------------------------
## Indicators of Compromise (IoCs)
- Network: high request rate or abnormally long-held connections from a single IP.
- Host: elevated CPU/memory/connection table usage tied to one source.
- Behavioral: service degradation correlating with sustained single-source activity.
- Log-based: web/applicati

Result 2
Source : ..\knowledge_base\DDoS.md
Attack : DDoS
--------------------------------------------------------------------------------
## Indicators of Compromise (IoCs)
- Network: sudden bandwidth spikes, high connection rates from geographically diverse IPs.
- Host: resource exhaustion (CPU, memory, connection tables) on target servers.
- Behavioral: sustained abnormal traffic volume disproportionate to normal usage patterns.
- L
Query: PortScan MITRE ATT&CK mapping

Result 1
Source : ..\knowledge_base\Po

### Verify Sources

Verify the source file associated with each loaded document.


In [19]:
for doc in docs:
    print(doc.metadata["source"])

..\knowledge_base\Web_Attack.md
